# PerfumeInsightLab
## Notebook 04 : Storytelling & Insights
--------------------------------------------------------------------  
#### Part of the multi-notebook EDA workflow : 01 (Data Overview) → 02 (Thematic EDA) → 03 (Quantitative EDA) → 04   
#### 1. Import librairies
#### 2. Load dataset   
#### 3. Feature engineering  
#### 4. Thematic insights & Trend analysis  
-  Evolution of Perfume Creation Over Time
-  Gender Evolution
-  Evolution of Olfactory Accords
-  Countries & Regional Dominance
-  Creative Landscape, Perfumers’ Influence
-  Consumer Ratings vs Characteristics
-  CONCLUSION

In [ ]:
# ------------------------------------------------------------------
# 1. Import libraries
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import unidecode

# ------------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------------
df = pd.read_csv("../data/fra_cleaned_v2.csv", encoding="ISO-8859-1", sep=";")

In [ ]:
# ------------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------------
# Weighted Rating calculation (corrected metric)

df_rated = df.dropna(subset=['Rating Value', 'Rating Count']).copy()      # Filter data to calculate the mean (C) and the threshold (m) accurately

                                                                          # Define the parameters for the weighted rating formula :
C = df_rated['Rating Value'].mean()              # C = Mean rating across the entire rated sample (N=509)
m = df_rated['Rating Count'].quantile(0.90)      # m = Minimum nber of votes required (using the 90th percentile of Rating Count)
m = max(100, m)                                  # Ensure 'm' is at least 100 for credibility

print(f"GLOBAL MEAN RATING (C) : {C:.2f}")       # print(f"Weighted Rating calculated. Global C: {C:.2f}, Threshold m: {m:.0f}")
print(f"MINIMUM VOTE THRESHOLD (m) : {m:.0f}")

def weighted_rating(row, m=m, C=C):                                       # Define the Weighted Rating function
    v = row['Rating Count']
    R = row['Rating Value']
    
    if pd.isna(v) or pd.isna(R):    # Handle NaN values for non-rated perfumes
        return np.nan
     
    return (v*R + m*C) / (v + m)    # Formula: (v*R + m*C) / (v+m)

df['weighted_rating'] = df.apply(weighted_rating, axis=1)                 # Apply the function to create the new column

print("\nTHE 'weighted_rating' COLUMN HAS BEEN CALCULATED ON THE ENTIRE DATAFRAME")
print("THIS COLUMN WILL NOW BE USED FOR ALL SUBSEQUENT PERFORMANCE ANALYSES")

top_weighted = df.sort_values('weighted_rating', ascending=False).head(10).round(3)
print("\nTOP 10 PERFUMES BASED ON WEIGHTED RATING (corrected for popularity bias) :\n")
display(top_weighted[['Perfume', 'Brand', 'Rating Value', 'Rating Count', 'weighted_rating']])

In [ ]:
# ------------------------------------------------------------------
# 4. Thematic insights & Trend analysis
# ------------------------------------------------------------------

In [ ]:
# Evolution of Perfume Creation Over Time
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# When did perfume creation accelerate ? Are modern perfumes (after 2000) more numerous but less rated ?

df_time = df[df['Year'] > 0].copy()
df_time['Decade'] = (df_time['Year']//10)*10

plt.figure(figsize=(10,4))                                                     # Perfume creation trend
sns.lineplot(x=df_time['Year'].value_counts().sort_index().index,
             y=df_time['Year'].value_counts().sort_index().values)
plt.title("PERFUME RELEASES OVER TIME (1780–2024)\n")
plt.xlabel("Year")
plt.ylabel("Nber of Perfumes")
plt.show()

df_time.groupby('Decade')['weighted_rating'].mean().plot(kind='bar', figsize=(10,4))
plt.title("AVERAGE WEIGHTED RATING BY DECADE\n")
plt.ylabel("Average Weighted Rating")
plt.show()

In [ ]:
# How has the volume of new perfume releases changed over time ?
# What is the historical relationship between the volume of production and the perceived quality of perfumes ?

df_time = df[df['Year'] > 1900].copy()                  
releases_per_year = df_time.groupby('Year')['Perfume'].nunique()                  # Calculate the volume of unique releases per year (Left Y-axis)
avg_rating_per_year = df_time.groupby('Year')['weighted_rating'].mean().round(2)  # Calculate the average weighted rating per year (Right Y-axis)

fig, ax1 = plt.subplots(figsize=(14, 6))          # Create the twin Y-axis plot

# Left Y-axis : Volume (bar chart)
ax1.bar(releases_per_year.index, releases_per_year.values, color='#1f77b4', label='Volume of Releases', alpha=0.7)
ax1.set_xlabel("Year")
ax1.set_ylabel("Volume of Unique Perfumes Released (Count)")
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.5)

# Right Y-axis : Quality (line plot)
orange = '#FF7F0E'                           
ax2 = ax1.twinx()
ax2.plot(avg_rating_per_year.index, avg_rating_per_year.values, color=orange, label='Average Weighted Rating', marker='o', linestyle='-', linewidth=2)
ax2.set_ylabel("Average Weighted Rating (Quality)")
ax2.tick_params(axis='y', labelcolor=orange)

ax2.set_ylim(min(avg_rating_per_year.min() - 0.01, 3.9),           # Adjusting limits to clearly show the small variations in rating
             max(avg_rating_per_year.max() + 0.01, 4.05)) 

plt.title("EVOLUTION OF PERFUME CREATION : VOLUME vs. AVERAGE QUALITY\n")
plt.tight_layout()
plt.show()

#### Evolution of perfume creation, dual Y-axis plot

This chart combines 2 distinct types of information on a single timeline : **volume** and **quality**.

| Element | Reference axis | It represents |
| ------- | -------------- | ------------- |
| 🟦 **Bars** | **Left Y-axis** (Volume of Releases) | The **absolute nber** of new perfumes launched each year → the scale of the industry's production. |
| 🟧 **Line** | **Right Y-axis** (Average Weighted Rating) | The **corrected average quality** for all perfumes launched in that year. This line is zoomed in to show slight variations in quality. |

This visualization tracks the industry's trajectory by comparing the **volume of new launches** with the **average perceived quality** over time.  

#### Key findings
🟦**Production boom :** The chart clearly shows a massive acceleration in the volume of unique perfume releases starting around the year **2000**.
The nber of new launches peaked significantly in the mid-to-late 2010s, demonstrating the shift towards a mass-market, fast-release industry model.   
🟧**Quality stability :** Despite this enormous increase in production volume, the average perceived quality has remained remarkably **stable**.

The perfume industry successfully **scaled up production without significantly diluting the average quality** of its output.   
While the market saw a flood of new releases, the weighted ratings show that the overall standard of new perfumes, on average, did not drop dramatically during this production boom.   
This suggests an efficient market capable of maintaining high-quality creation despite high launch frequency.   

In [ ]:
# Gender Evolution 
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# Has the industry become more unisex ? Which decades show the biggest change ?

gender_by_year = (
    df_time.groupby(['Year','Gender'])
    .size().unstack(fill_value=0).rolling(5).mean()        # smoothing
)

gender_by_year.plot.area(figsize=(12,5), colormap='Set2')
plt.title("EVOLUTION OF PERFUME GENDER DISTRIBUTION (rolling 5-year average)\n")
plt.xlabel("Year")
plt.ylabel("Nber of Perfumes")
plt.show()

In [ ]:
# Evolution of Olfactory Accords
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# Are woody or citrus accords more frequent today ? Are floral accords declining ?

accord_cols = ['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5']

df_accords = df_time.melt(id_vars=['Year'], value_vars=accord_cols, var_name='Rank', value_name='Accord')

df_accords['Accord'] = df_accords['Accord'].str.lower().str.strip()    # Clean the accord names
top_accords = ['woody','floral','citrus','sweet','aromatic']

df_accords_filtered = df_accords[                           
    df_accords['Accord'].isin(top_accords)
].dropna(subset=['Accord'])

# Key correction : group and count occurrences per Year → how many times each accord appears in each year ?
df_accord_counts = df_accords_filtered.groupby(['Year', 'Accord']).size().reset_index(name='Nber_of_Occurrences')

plt.figure(figsize=(10, 6)) 
sns.lineplot(
    data=df_accord_counts,
    x='Year',
    y='Nber_of_Occurrences',    # Correct Y-axis: now a numeric count
    hue='Accord',               # Legend based on Accord name
    errorbar=None
)

plt.title("EVOLUTION OF TOP ACCORDS OVER TIME\n")
plt.ylabel("Nber of Occurrences")
plt.xlabel("Year")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# What are the most popular and frequent volatile components (Top Notes) used to define a perfume's 1st impression ?

# Data preparation : extract and clean Top Notes
notes_text_top = df['Top Notes'].astype(str).str.lower().str.cat(sep=',')
notes_text_top = notes_text_top.replace(',', ' ')                  

# Define stop words : common terms that might dominate the cloud but lack descriptive value
stop_words_top = ['nan', 'citrus', 'fruit', 'notes', 'fresh', 'spicy'] 

wordcloud_top = WordCloud(
    width=800, height=400, background_color='white', colormap='plasma', max_words=100, stopwords=set(stop_words_top), collocations=False
).generate(notes_text_top)

plt.figure(figsize=(9, 5))
plt.imshow(wordcloud_top, interpolation='bilinear')
plt.axis("off")
plt.title("DOMINANT TOP NOTES\n")
plt.show()

#### Olfactory trends
This Wordcloud visualizes the most frequent **Top Notes**, the highly volatile ingredients that form the perfume's initial impression and opening statement.

**Contrast with Base Notes :** Unlike the heavier, long-lasting ingredients seen in the Base Notes Wordcloud, the most common Top Notes are typically light, energetic and highly volatile (e.g., various citruses, fresh herbs or light spices).    
**Initial impact :** The size of the words indicates which ingredients are most frequently used by perfumers to capture the consumer's attention immediately upon application. They define the perfume's category and initial emotional impact.   
**Key volatile materials :** Notes like **Bergamot**, **Lemon**, **Pink Pepper**, **Grapefruit**, often dominate the opening phase of a fragrance.

In [ ]:
# What are the most frequent core components (Middle Notes) used by the industry to define the central character and transition phase of a fragrance ?

notes_text_middle = df['Middle Notes'].astype(str).str.lower().str.cat(sep=',')
notes_text_middle = notes_text_middle.replace(',', ' ')                  # Replace commas with spaces so WordCloud treats notes as individual words

stop_words_middle = ['nan', 'floral', 'notes', 'herbaceous', 'watery']   # Define stop words : generic terms

wordcloud_middle = WordCloud(
    width=800, height=400, background_color='white', colormap='cividis', max_words=100, stopwords=set(stop_words_middle), collocations=False
).generate(notes_text_middle)

plt.figure(figsize=(9, 5))
plt.imshow(wordcloud_middle, interpolation='bilinear')
plt.axis("off")
plt.title("DOMINANT ♥ NOTES\n")
plt.show()

#### Olfactory trends
The most frequent **Middle Notes** (♥ Notes). These ingredients define the true character of the fragrance, emerging after the top notes fade and holding the composition together before the base notes appear.

**Defining the character :** The frequency of these notes reveals the current trends in the core structure of modern perfumery. Middle Notes are typically composed of heavier florals, spices or green notes.   
**Transition and balance :** The dominant words here show the materials most often chosen by perfumers to create a balanced, lasting bridge between the energetic opening (Top Notes) and the deep longevity (Base Notes).  
**Cross-sectional view :** Comparing this visualization with the Top and Base Notes Wordclouds provides a full, 3-dimensional understanding of current market preferences across all phases of a fragrance's life cycle.   

In [ ]:
# What are the most frequent and dominant olfactory components (Base Notes) used by the industry to ensure scent longevity ?

notes_text = df['Base Notes'].astype(str).str.lower().str.cat(sep=',') # Convert the 'Base Notes' column to a single string, handling NaNs and lowercasing                          
notes_text = notes_text.replace(',', ' ')                              # Replace commas with spaces so WordCloud treats notes as individual words

stop_words = ['nan', 'musk', 'wood', 'base', 'notes', 'amber', 'sweet', 'citrus'] 

wordcloud = WordCloud(
    width=800, height=400, background_color='white', colormap='viridis', max_words=100, stopwords=set(stop_words), collocations=False 
).generate(notes_text)                                    # collocation=False avoids combining words that appear together (e.g., 'white' and 'floral')

plt.figure(figsize=(9, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("DOMINANT BASE NOTES\n")
plt.show()

#### Olfactory trends
A quick visual summary of the most frequently occurring **Base Notes** used in the perfume industry.   
Base Notes form the deep, lasting foundation of a fragrance. Their frequency is a key indicator of **long-term olfactory trends** and consumer preferences for scent longevity and dry-down.

**Size = Frequency :** The larger the word, the higher the frequency of that ingredient across the entire database of perfumes.   
**Key Materials :** The visualization highlights the essential, long-lasting materials that perfumers commonly rely on. Dominant notes like **Vetiver**, **Patchouli**, **Sandalwood**, specific types of **Vanilla** or **Tonka Bean**.  
**Filtering for clarity :** Generic, overused terms (like 'musk' and 'wood') were added to a stop-word list to prevent them from dominating the cloud, ensuring that more descriptive and strategically important notes are visible.

In [ ]:
# Countries & Regional Dominance
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# Which countries dominate perfume creation across time ? Has France remained the leader ?

country_trends = df_time.groupby(['Year','Country']).size().unstack(fill_value=0)
top_countries = country_trends.sum().sort_values(ascending=False).head(5).index
country_trends[top_countries].rolling(3).mean().plot(figsize=(9,5))
plt.title("PERFUME CREATION BY TOP 5 COUNTRIES (1780–2024)\n")
plt.xlabel("Year")
plt.ylabel("Nber of Perfumes")
plt.show()

In [ ]:
# Do the countries that produce the most perfume also produce the highest quality perfume ?

country_counts = df.groupby('Country')['Perfume'].count()

min_country_volume = 50
qualified_countries = country_counts[country_counts >= min_country_volume].index

df_qualified_countries = df[df['Country'].isin(qualified_countries)].copy()

country_avg_rating = (df_qualified_countries.groupby('Country')['weighted_rating'].mean().sort_values(ascending=False).head(10).round(3))

plt.figure(figsize=(9, 4))               
sns.barplot(x=country_avg_rating.values, y=country_avg_rating.index, hue=country_avg_rating.index, palette='viridis', edgecolor='black')               
                                                    # assign the categorical variable 'Country' to 'hue' to satisfy the new Seaborn API
plt.legend([],[], frameon=False) 

plt.title(f"TOP 10 COUNTRIES BY AVERAGE WEIGHTED RATING (min {min_country_volume} perfumes)\n")
plt.xlabel("Average Weighted Rating")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

#### Geographical dominance : Quality vs. Volume

This chart ranks countries based on the **average weighted rating** of their perfumes, using only countries with a significant volume of releases (min. 50 perfumes).   
This crucial filter allows us to identify global leaders in **quality**, rather than just **volume**.

**Challenging assumptions :** By using the weighted rating, we look past simple production quantity. The results often reveal that countries typically dominant in **volume** (e.g., USA, France) may not necessarily top the list when judged purely on the average perceived **quality** of their offerings.   
**The power of niche :** Countries ranking highest in this chart often represent powerful **niche markets** or established houses known for consistently high customer satisfaction, demonstrating excellence and quality control over mass-market appeal.

In [ ]:
# Creative Landscap, Perfumers’ Influence
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# Which perfumers have the largest creative footprint ? Are they associated with multiple brands ?

top_perf = df['Perfumer'].value_counts().head(10)
sns.barplot(y=top_perf.index, x=top_perf.values)
plt.title("TOP 10 MOST PRODUCTIVE PERFUMERS\n")
plt.xlabel("Nber of Perfumes")
plt.show()

perf_brand = (
    df[['Perfumer','Brand']]
    .drop_duplicates().groupby('Perfumer').nunique().sort_values(by='Brand', ascending=False).head(10)
)
sns.barplot(y=perf_brand.index, x=perf_brand['Brand'])
plt.title("PERFUMERS WITH THE WIDEST BRAND REACH\n")
plt.xlabel("Nber of Brands")
plt.show()

In [ ]:
# Among the most prolific creators, which perfumers consistently deliver the highest-rated perfumes ?

min_perfumer_volume = 5                                                # Set min volume to 5 to ensure a robust sample after filtering unknowns
unknown_values = ['unknown', 'Unknown', 'N/A', 'n/a', 'nan', '']
                                                    
df_perfumer = df[['Perfumer', 'weighted_rating']].dropna().copy()      # Data preparation : explode the perfumers column

# Split the names and create one row per Perfumer (Corrected typos)
df_perfumer_exploded = df_perfumer.assign(Perfumer=df_perfumer['Perfumer'].str.split(',')).explode('Perfumer')
df_perfumer_exploded['Perfumer'] = df_perfumer_exploded['Perfumer'].str.strip() 

df_qualified_perfumers_initial = df_perfumer_exploded[                 # Filter out unknown perfumers and empty strings
    ~df_perfumer_exploded['Perfumer'].str.lower().isin(unknown_values) &       # Remove 'Unknown' and similar tags (case-insensitive)
    (df_perfumer_exploded['Perfumer'] != '')                                   # Remove any explicit empty strings 
]
                                                                       # Filter Qualified Perfumers (at least 5 creations)
perfumer_counts = df_qualified_perfumers_initial.groupby('Perfumer')['weighted_rating'].count()
qualified_perfumers = perfumer_counts[perfumer_counts >= min_perfumer_volume].index

df_qualified_perfumers = df_qualified_perfumers_initial[df_qualified_perfumers_initial['Perfumer'].isin(qualified_perfumers)]

num_qualified = len(qualified_perfumers)                               # --- Verification step : check the count of qualified perfumers ---
print(f"NBER of QUALIFIED PERFUMERS REMAINING (min {min_perfumer_volume} creations) : {num_qualified}\n")

if df_qualified_perfumers.empty:                                       # --- Continue only if not empty ---
    print(f"ERROR : Filtered DataFrame is still EMPTY. The data likely lacks sufficient identified creators.")
else:    
    perfumer_avg_rating = (                       # Calculate Average Weighted Rating and Rank
        df_qualified_perfumers.groupby('Perfumer')['weighted_rating']
        .mean().sort_values(ascending=False).head(10).round(3)
    )
                                                
    plt.figure(figsize=(8, 3))                 
    sns.barplot(x=perfumer_avg_rating.values, y=perfumer_avg_rating.index, hue=perfumer_avg_rating.index, palette='viridis', edgecolor='black')

    plt.legend([],[], frameon=False) 
    plt.title(f"TOP 10 PERFUMERS BY AVERAGE WEIGHTED RATING\n")
    plt.xlabel("Average Weighted Rating")
    plt.ylabel("Perfumer")
    plt.tight_layout()
    plt.show()

    print("TOP 10 PERFUMERS BY QUALITY (weighted rating) :")
    print(perfumer_avg_rating)

#### Identifying quality leaders (top perfumers)

This script identifies the industry's most consistent **"hit-makers"** by analyzing the **average weighted rating** of their entire portfolio.

1.  **Explosion and Cleaning :** We first split perfumer names (as a single perfume can have multiple creators) and clean the data by filtering out **'unknown'** or empty entries, which would bias the average.
2.  **Volume filter :** A minimum volume threshold of **5 creations** (`min_perfumer_volume = 5`) is set. This is crucial to ensure that the average rating for each perfumer is based on a robust sample, eliminating high scores from individuals with only 1 or 2 products.
3.  **Ranking :** The mean `weighted_rating` is calculated for all qualified perfumers, and the **top 10** are ranked and visualized.

This methodology guarantees that the ranking reflects sustained excellence, not just market saturation or single, unrepresentative successes.

#### Top perfumers by average weighted rating
**Consistency is key :** The perfumers listed here are the masters of **consistent quality**.   
Their high average scores demonstrate their ability to create compositions that resonate positively with consumers over a significant body of work.   
**Quality vs. Volume of influence :** This ranking contrasts with a simple count of creations, where the most frequent creators often see their average rating diluted.   
The perfumers at the top maintain high standards, suggesting superior skill and creative consistency.   
**Focus on the score :** Since the `weighted_rating` for top performers is very tight, observe the precise ranking.   
A small difference on this scale indicates a major lead in consistent perceived quality.   

In [ ]:
# Consumer Ratings vs Characteristics
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# Do certain accords or genres score higher ? Are older perfumes rated better ?

for accord in ['woody','floral','sweet']:
    avg_rating = df[df[accord_cols].apply(lambda x: x.str.contains(accord, case=False).any(), axis=1)]['weighted_rating'].mean()
    print(f"AVERAGE WEIGHTED RATING FOR {accord} PERFUMES : {avg_rating:.2f}\n")

sns.scatterplot(x='Year', y='weighted_rating', data=df_time, alpha=0.3)             
plt.title("WEIGHTED RATING vs YEAR OF RELEASE\n") 
plt.show()

In [ ]:
# Which brands maintain the highest level of average perceived quality across their entire portfolio ?

min_brand_volume = 5                                              # Reducing the minimum volume per brand to ensure at least 10 brands are included
unknown_values = ['unknown', 'Unknown', 'N/A', 'n/a', 'nan', '']

df_brand = df[['Brand', 'weighted_rating']].dropna().copy()                             # Filter out brands without rating data and unknown brands

df_brand_filtered = df_brand[                                     # Filter out unknown/empty brand names
    ~df_brand['Brand'].str.lower().isin(unknown_values) &
    (df_brand['Brand'].str.strip() != '')
]
                                                                 
brand_counts = df_brand_filtered.groupby('Brand')['weighted_rating'].count()             # Filter qualified brands (at least 5 releases)
qualified_brands = brand_counts[brand_counts >= min_brand_volume].index

df_qualified_brands = df_brand_filtered[df_brand_filtered['Brand'].isin(qualified_brands)]

num_qualified_brands = len(qualified_brands)                      # --- Verification step ---
print(f"NBER of QUALIFIED BRANDS REMAINING (min {min_brand_volume} releases): {num_qualified_brands}\n")

if df_qualified_brands.empty:                                     # --- Continue only if not empty ---
    print(f"ERROR : Filtered Brands DataFrame is EMPTY. Try reducing the 'min_brand_volume' threshold even further.")    
else:
    brand_avg_rating = (                                                                # Calculate average weighted rating and rank
        df_qualified_brands.groupby('Brand')['weighted_rating']
        .mean().sort_values(ascending=False).head(10).round(4)  
    )
   
    plt.figure(figsize=(9, 3))                                  
    sns.barplot(x=brand_avg_rating.values, y=brand_avg_rating.index, hue=brand_avg_rating.index, palette='magma', edgecolor='black')

    plt.legend([],[], frameon=False) 
    plt.title(f"TOP 10 BRANDS BY AVERAGE WEIGHTED RATING (min {min_brand_volume} releases)\n", fontsize=14)
    plt.xlabel("Average Weighted Rating")
    plt.ylabel("Brand")
    plt.tight_layout()
    plt.show()

    print("TOP 10 BRANDS BY QUALITY (weighed rating):")
    print(brand_avg_rating)

#### Identifying quality leaders (top brands)
This script analyzes **which brands consistently deliver the best quality**, measured by the **average weighted rating**.  

1.  **Data cleaning :** Brands with unknown names and missing rating data, are filtered out.  
3.  **Volume filter :** To ensure that the average rating is statistically stable and not based on just 1 or 2 successful perfumes, only brands that have launched at least **5 fragrances** (`min_brand_volume = 5`) are included.  
5.  **Ranking :** The `weighted_rating` is calculated for all eligible brands. The **top 10** are ranked.  

This methodology ensures that the ranking highlights sustained quality excellence, not just market size or occasional success.

#### Top brands 
**The power of quality :** The ranking is based purely on the average weighted rating, which corrects for the bias toward popular, highly-voted perfumes.    
Brands appearing here are confirmed as maintaining high standards across their qualified releases.   
**Niche vs. Legacy :** Observe whether the top spots are dominated by small, niche houses (suggesting tight quality control and dedicated fan bases) or by established, high-volume legacy brands (suggesting successful maintenance of quality despite large-scale production).   
**Actionable insight :** The average scores for the top 7 brands are very close.    
Even small differences in the weighted rating are significant, distinguishing the absolute best in consistent quality.

In [ ]:
# Which specific olfactory accords consistently achieve the highest average perceived quality from consumers ?

# Data preparation : check for the main accord column and calculate means → to group the data by a main olfactory classification column
# If a direct column like 'Olfactory Group' exists, it's used ; otherwise, it falls back to keyword searching.

try:
    # Attempt 1: Group by a dedicated classification column (e.g., 'Olfactory Group')
    # Please ensure this column name is correct in your DataFrame.
    df_accords = df[['weighted_rating', 'Olfactory Group']].dropna().copy()   
    
    accord_quality = (                                                          # Calculate the mean weighted rating for each group
        df_accords.groupby('Olfactory Group')['weighted_rating']
        .mean().sort_values(ascending=False).round(3)
    )

except KeyError:            # Fallback method: If the specific column is not found, search for main accords by keyword (more robust but less precise)   
    accords_cles = ['woody', 'floral', 'sweet', 'citrus', 'amber', 'spicy']     # Define key accords to search for
    resultats_qualite_accords = {}

    # Identify a column to search in (e.g., the first column containing the word 'accord')
    accord_col = [col for col in df.columns if 'accord' in col.lower()][0] if any('accord' in col.lower() for col in df.columns) else 'Perfume' 
    
    for accord in accords_cles:
        # Filter rows where the accord is present in the text column
        df_filtered = df[df[accord_col].astype(str).str.lower().str.contains(accord, na=False)]
        
        if not df_filtered.empty:
            resultats_qualite_accords[accord] = df_filtered['weighted_rating'].mean().round(4)
            
    accord_quality = pd.Series(resultats_qualite_accords).sort_values(ascending=False)
    
if not accord_quality.empty:                                             
    accord_quality_top = accord_quality.head(10)                                 # Filter the top 10 (or less if there are fewer categories)
    
    plt.figure(figsize=(9, 3))
    sns.barplot(x=accord_quality_top.values, y=accord_quality_top.index, hue=accord_quality_top.index, palette='mako', edgecolor='black')

    plt.legend([],[], frameon=False)                                             # Hide the redundant legend
    plt.title("AVERAGE WEIGHTED RATING BY OLFACTORY ACCORD\n")
    plt.xlabel("Average Weighted Rating")
    plt.ylabel("Olfactory Accord") 
    plt.xlim(accord_quality_top.min() - 0.005, accord_quality_top.max() + 0.005) # Zoom the X-axis to clearly show the small variance in rating scores
    plt.tight_layout()
    plt.show()

    print("AVERAGE WEIGHTED RATING BY OLFACTORY ACCORD (TOP) :")
    print(accord_quality_top)
else:
    print("\nERROR : Cannot calculate accord averages. Please verify the name of the olfactory accord column")

#### Consumer ratings vs. characteristics : the quality of olfactory accords
This analysis links **olfactory groups/accords** directly to the **average perceived quality** (`weighted_rating`).   
Since the `weighted_rating` corrects for simple popularity, this ranking reveals which scent families are **consistently rated higher** by consumers.

**Key question :** Which scent profiles inherently command a higher quality score ?  
**Actionable insight :** The results highlight whether classic, complex families (e.g., woody, floral) or more modern, popular groups (e.g., sweet, fruity) achieve the highest average quality scores.   
This insight can guide future product development efforts by focusing on the profiles that resonate strongest with quality perception.   
**Tight variance :** Note that the differences between the top-ranking accords are usually minimal (often a fraction of a point).   
Even small leads are significant in the context of consistent quality across thousands of perfumes.

---
---

### CONCLUSION : Market dynamics and quality leaders

This EDA of the perfume industry revealed several critical dynamics regarding production, quality perception and creative trends :

#### 1. Volume vs. Quality dynamics  
The industry experienced a massive **production boom** starting post-2000.   
Crucially, this explosion in quantity **did not drastically dilute the average perceived quality** of perfumes, which remained relatively stable.   

#### 2. Identifying quality leaders  
**Geographical dominance :** Quality leadership often shifts away from sheer volume producers (like the US/France) to smaller markets or exclusive regions when ranked by weighted rating, highlighting the strength of **niche quality control** (a sustained high average quality achieved by smaller producers who prioritize superior product standards over mass-market volume, resulting in consistently higher corrected ratings).     
**Creative excellence :** The analysis identified specific **Perfumers** and **Brands** that consistently achieve the highest average quality scores across their entire portfolio, confirming their **creative consistency** and skill.   

#### 3. Olfactory trends & consumer preference
**Scent structure :** The **Wordclouds** confirmed that **Top Notes** prioritize highly volatile citruses for instant impact, while **Base Notes** rely on heavy, long-lasting materials (e.g., woody notes, patchouli) for structural longevity.   
**Accords vs. Ratings :** This analysis reveals which scent families (e.g., woody, amber, floral) consistently achieve the highest average perceived quality, guiding future development efforts.   

#### Key takeaways 
* A clear **boom in perfume creation** commenced after the year 2000
* A visible **shift toward unisex fragrances**
* **“Woody” and “Sweet” accords** are gaining ground over more traditional **“Floral”** ones
* **France** remains the historical leader in overall perfume creation volume
* A small number of **perfumers** shape a vast share of the market's creative landscape